# Part 1 - 실습 2: 자연어처리 기초 & 프롬프트 엔지니어링
**소요시간: 50분** | 난이도: ⭐⭐⭐

## 학습 목표
1. Zero-shot / Few-shot / Chain-of-Thought 프롬프팅 기법을 이해합니다.
2. System Prompt로 AI 페르소나와 출력 형식을 제어합니다.
3. Structured Output(JSON 응답) 기법을 실습합니다.

## 프롬프트 엔지니어링 핵심 기법
| 기법 | 설명 | 언제 사용 |
|---|---|---|
| **Zero-shot** | 예시 없이 지시만으로 태스크 수행 | 간단하고 명확한 작업 |
| **Few-shot** | 2~5개 예시를 함께 제공 | 특정 형식/스타일이 필요할 때 |
| **Chain-of-Thought** | "단계별로 생각하라" 지시 | 추론·계산이 필요한 복잡한 문제 |
| **System Prompt** | 역할·규칙·제약을 사전 정의 | 일관된 AI 행동 제어 |
| **Structured Output** | JSON 형식 응답 강제 | 프로그램에서 파싱할 때 |


---
## 🏢 기업 시나리오 — 추가 개발 없이 업무 자동화

당신은 **데이터 분석가**입니다. 모델을 학습시키지 않고, **프롬프트만 잘 써서** 업무를 자동화합니다.

이번 실습에서는 다음을 익힙니다.
1. **Zero/Few-shot** → 리뷰·문의 자동 분류
2. **Chain-of-Thought** → 복잡한 계산·심사 로직을 단계별로
3. **Structured Output(JSON)** → 시스템에 바로 넣을 수 있는 데이터로 추출
4. **System Prompt** → 일관된 회사 톤·페르소나 제어

> 💡 실무 핵심은 **Structured Output**입니다. 응답을 JSON으로 받으면 그대로 DB·API에 연결됩니다. 예: 고객 문의 → {유형, 긴급도, 담당부서} 자동 태깅.


In [1]:
# ✅ [제공 코드] 환경 초기화 & 헬퍼 함수
import boto3, json

bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-1')
MODEL_ID = 'us.anthropic.claude-sonnet-4-20250514-v1:0'

def claude(user_prompt, system_prompt=None, temperature=0.7, max_tokens=1024):
    """Claude converse API 래퍼"""
    kwargs = {
        'modelId': MODEL_ID,
        'messages': [{'role': 'user', 'content': [{'text': user_prompt}]}],
        'inferenceConfig': {'maxTokens': max_tokens, 'temperature': temperature}
    }
    if system_prompt:
        kwargs['system'] = [{'text': system_prompt}]
    resp = bedrock_runtime.converse(**kwargs)
    return resp['output']['message']['content'][0]['text']

print('✅ claude() 헬퍼 함수 준비 완료')


✅ claude() 헬퍼 함수 준비 완료


## ✏️ TODO 1: Zero-shot vs Few-shot 비교

동일한 감성 분류 태스크를 Zero-shot과 Few-shot으로 각각 수행하고 결과를 비교하세요.


In [2]:
# ✏️ TODO 1: Zero-shot vs Few-shot 프롬프트를 작성하고 결과를 비교하세요
test_reviews = [
    '이 강의 정말 유익했어요. 내용도 알차고 강사님도 친절하세요.',
    '과제가 너무 많고 설명이 부족해서 따라가기 힘들었습니다.',
    '그냥 평범한 수업이에요. 특별히 좋거나 나쁘지 않았습니다.',
]

# Zero-shot 프롬프트
zero_shot_template = """
다음 리뷰의 감성을 분석하여 '긍정', '부정', '중립' 중 하나만 답하세요.
리뷰: {review}
"""

# Few-shot 프롬프트 (예시 포함)
few_shot_template = """
다음은 강의 리뷰 감성 분류 예시입니다.

예시:
리뷰: '최고의 강의입니다! 강력 추천해요.'  → 긍정
리뷰: '돈이 아까웠습니다. 전혀 도움 안 됨.'  → 부정
리뷰: '수업 내용은 보통이었어요.'  → 중립

이제 다음 리뷰를 분류하세요.
리뷰: '{review}'  →
"""

print(f"{'리뷰':<30} {'Zero-shot':>12} {'Few-shot':>10}")
print('-' * 60)
for review in test_reviews:
    # Zero-shot 호출
    zero_result = claude(
        zero_shot_template.format(review=review),   # ← review=review
        temperature=0.1                          # ← 0.1 (일관된 분류를 위해 낮게)
    ).strip()

    # Few-shot 호출
    few_result = claude(
        few_shot_template.format(review=review),    # ← review=review
        temperature=0.1                          # ← 0.1
    ).strip()

    print(f"{review[:28]:<30}| {zero_result:>12} | {few_result:>10}")


리뷰                                Zero-shot   Few-shot
------------------------------------------------------------


이 강의 정말 유익했어요. 내용도 알차고 강사님도   |           긍정 | 리뷰: '이 강의 정말 유익했어요. 내용도 알차고 강사님도 친절하세요.' → **긍정**

이 리뷰는 다음과 같은 긍정적 표현들을 포함하고 있습니다:
- "정말 유익했어요" - 강의의 가치를 높게 평가
- "내용도 알차고" - 강의 내용에 대한 만족
- "강사님도 친절하세요" - 강사에 대한 호의적 평가

전반적으로 강의와 강사에 대해 만족스러워하는 긍정적인 감성을 나타냅니다.


과제가 너무 많고 설명이 부족해서 따라가기 힘들었습  |           부정 | 리뷰: '과제가 너무 많고 설명이 부족해서 따라가기 힘들었습니다.' → **부정**

이 리뷰는 "과제가 너무 많고", "설명이 부족", "따라가기 힘들었다"는 부정적인 표현들로 구성되어 있어 강의에 대한 불만을 명확히 드러내고 있습니다.


그냥 평범한 수업이에요. 특별히 좋거나 나쁘지 않았  |           중립 | 리뷰: '그냥 평범한 수업이에요. 특별히 좋거나 나쁘지 않았습니다.' → **중립**

이 리뷰는 "그냥 평범한", "특별히 좋거나 나쁘지 않았다"는 표현을 통해 강의에 대한 명확한 긍정적 또는 부정적 감정을 드러내지 않고 있어 중립적 감성으로 분류됩니다.


## ✏️ TODO 2: Chain-of-Thought 프롬프팅

복잡한 문제를 단계별로 추론하도록 Chain-of-Thought 프롬프트를 작성하세요.


In [7]:
# ✏️ TODO 2: CoT 프롬프트를 작성하여 단계별 추론을 유도하세요
problem = """
학교 AI 강의 수강생이 20명이다.
첫 번째 과제를 제출한 학생은 전체의 80%이다.
그 중 60%가 두 번째 과제도 제출했다.
두 번째 과제를 제출하지 않은 학생은 몇 명인가?
"""

# 일반 프롬프트 (CoT 없음)
normal_prompt = f'다음 문제를 풀어주세요: {problem}'

# CoT 프롬프트 — 빈칸 완성 --problem
cot_prompt = f"""
다음 문제를 단계별로 풀어주세요.

규칙:
1. 각 계산 단계를 [단계 N] 형식으로 표시하세요.
2. 각 단계에서 수식과 결과를 명시하세요.
3. 마지막에 [최종 답변]을 반드시 제시하세요.

문제: {problem}
"""

print('=== 일반 프롬프트 결과 ===')
print(claude(normal_prompt, temperature=0.1))
print()
print('=== Chain-of-Thought 결과 ===')
print(claude(cot_prompt, temperature=0.1))   # ← cot_prompt


=== 일반 프롬프트 결과 ===


주어진 정보를 정리하여 단계별로 풀어보겠습니다.

**주어진 정보:**
- 전체 수강생: 20명
- 첫 번째 과제 제출률: 80%
- 첫 번째 과제 제출자 중 두 번째 과제도 제출한 비율: 60%

**단계별 풀이:**

1) **첫 번째 과제를 제출한 학생 수**
   - 20명 × 80% = 20 × 0.8 = 16명

2) **두 번째 과제를 제출한 학생 수**
   - 첫 번째 과제 제출자 중 60%가 두 번째 과제도 제출
   - 16명 × 60% = 16 × 0.6 = 9.6명 → **10명** (반올림)

3) **두 번째 과제를 제출하지 않은 학생 수**
   - 전체 학생 수 - 두 번째 과제 제출 학생 수
   - 20명 - 10명 = **10명**

따라서 두 번째 과제를 제출하지 않은 학생은 **10명**입니다.

=== Chain-of-Thought 결과 ===


주어진 문제를 단계별로 해결하겠습니다.

**[단계 1] 첫 번째 과제를 제출한 학생 수 계산**
- 전체 학생 수: 20명
- 첫 번째 과제 제출률: 80%
- 첫 번째 과제를 제출한 학생 수 = 20 × 0.8 = 16명

**[단계 2] 두 번째 과제를 제출한 학생 수 계산**
- 첫 번째 과제를 제출한 학생 중 두 번째 과제도 제출한 비율: 60%
- 두 번째 과제를 제출한 학생 수 = 16 × 0.6 = 9.6명
- 학생 수는 정수이므로 = 10명 (반올림) 또는 9명 (버림)

**[단계 3] 두 번째 과제를 제출하지 않은 학생 수 계산**
- 전체 학생 수에서 두 번째 과제를 제출한 학생 수를 뺌
- 9.6명을 정확히 계산하면: 16 × 0.6 = 9.6명

실제로는 학생 수가 소수점이 나올 수 없으므로, 문제의 조건을 다시 해석하면:
- 16명 중 60%가 두 번째 과제를 제출 = 16 × 0.6 = 9.6명
- 가장 가까운 정수로 처리하면 10명이 두 번째 과제를 제출

**두 번째 과제를 제출하지 않은 학생 수 = 20 - 10 = 10명**

**[최종 답변]**
두 번째 과제를 제출하지 않은 학생은 **10명**입니다.


## ✏️ TODO 3: Structured Output — JSON 응답 강제

AI가 항상 파싱 가능한 JSON으로 응답하도록 프롬프트를 설계하세요.


In [8]:
# ✏️ TODO 3: JSON 형식 응답을 강제하는 System Prompt를 작성하세요
SYSTEM_JSON = """
당신은 텍스트 분석 AI입니다.
반드시 다음 JSON 형식으로만 응답하세요. JSON 외 다른 텍스트는 절대 포함하지 마세요.

{
  "sentiment": "긍정|부정|중립",
  "confidence": 0.0~1.0,
  "keywords": ["키워드1", "키워드2", "키워드3"],
  "summary": "한 문장 요약"
}
"""

texts = [
    '오늘 딥러닝 강의를 들었는데 내용이 너무 어려워서 힘들었지만 끝내고 나니 뿌듯했다.',
    '과제 제출 마감이 너무 촉박해서 밤을 새웠다. 다음엔 여유 있게 시작해야겠다.',
]

for text in texts:
    raw = claude(
        user_prompt=text,          # ← text
        system_prompt=SYSTEM_JSON,        # ← SYSTEM_JSON
        temperature=0.1           # ← 0.1 (JSON 일관성을 위해)
    )
    try:
        parsed = json.loads(raw) # ← raw
        print(f"텍스트: {text[:30]}...")
        print(f"  감성: {parsed['sentiment']}  신뢰도: {parsed['confidence']}")   # ← 'sentiment'
        print(f"  키워드: {parsed['keywords']}")
        print(f"  요약: {parsed['summary']}\n")
    except json.JSONDecodeError:
        print(f'JSON 파싱 실패:\n{raw}')


텍스트: 오늘 딥러닝 강의를 들었는데 내용이 너무 어려워서 힘들...
  감성: 긍정  신뢰도: 0.7
  키워드: ['딥러닝', '강의', '어려움', '뿌듯함']
  요약: 딥러닝 강의가 어려웠지만 완주 후 성취감을 느꼈다.



텍스트: 과제 제출 마감이 너무 촉박해서 밤을 새웠다. 다음엔 ...
  감성: 부정  신뢰도: 0.7
  키워드: ['과제', '마감', '밤샘', '여유']
  요약: 과제 마감에 쫓겨 밤을 새우며 다음엔 미리 시작하겠다고 다짐하는 상황



## ✏️ TODO 4: System Prompt로 페르소나 제어

학교 AI 강의 조교 페르소나를 System Prompt로 정의하고, 동일 질문에 대해 다른 페르소나와 응답을 비교하세요.


In [9]:
# ✏️ TODO 4: 두 가지 System Prompt 페르소나를 정의하고 응답을 비교하세요
SYSTEM_TUTOR = """
당신은 대학교 AI 강의 조교입니다.
다음 규칙을 반드시 지키세요:
- 친절하고 격려하는 톤으로 답변하세요.
- 개념 설명 시 현실 예시를 반드시 포함하세요.
- 학생이 이해하기 어렵다고 하면 더 쉬운 비유를 사용하세요.
- 답변 끝에 관련 심화 학습 질문을 하나 제안하세요.
"""

SYSTEM_EXPERT = """
당신은 AI 연구소 선임 연구원입니다.
다음 규칙을 반드시 지키세요:
- 기술적으로 정확하고 간결하게 답변하세요.
- 전문 용어를 사용하되 필요시 영문 원어를 병기하세요.
- 학술 논문 또는 공식 문서 기준으로 설명하세요.
"""

question = 'Transformer 모델의 Attention 메커니즘이 무엇인가요?'

print('=== 강의 조교 페르소나 ===')
print(claude(question, system_prompt=SYSTEM_TUTOR))   # ← SYSTEM_TUTOR
print()
print('=== 연구원 페르소나 ===')
print(claude(question, system_prompt=SYSTEM_EXPERT))   # ← SYSTEM_EXPERT


=== 강의 조교 페르소나 ===


안녕하세요! Attention 메커니즘에 대해 친절하게 설명드리겠습니다. 😊

## Attention 메커니즘이란?

Attention은 **"중요한 정보에 집중하는"** 메커니즘입니다. 마치 우리가 책을 읽을 때 핵심 문장에 형광펜을 치는 것처럼, 모델이 입력 데이터에서 가장 관련성 높은 부분에 "주의"를 기울이는 방식이에요.

## 현실 예시로 이해하기

**번역 상황을 생각해보세요:**
- 영어: "The cat sat on the mat"
- 한국어: "고양이가 매트 위에 앉았다"

"고양이가"를 번역할 때, 모델은 영어 문장에서 "cat"에 가장 높은 attention을 주고, "The", "sat", "on" 등에는 낮은 attention을 줍니다.

## Self-Attention의 핵심 과정

1. **Query(질문), Key(열쇠), Value(값)** 생성
   - 각 단어가 다른 모든 단어와의 관계를 계산합니다

2. **Attention Score 계산**
   - 어떤 단어끼리 더 관련이 있는지 점수를 매깁니다

3. **가중합 계산**
   - 중요한 정보는 더 크게, 덜 중요한 정보는 더 작게 반영합니다

## 왜 혁신적인가?

기존 RNN과 달리 **모든 위치의 정보를 동시에** 처리할 수 있어서:
- ⚡ 병렬 처리가 가능해 훨씬 빠름
- 🎯 장거리 의존성도 효과적으로 포착
- 🔍 어떤 부분에 집중했는지 해석 가능

정말 멋진 개념이죠! 이해가 잘 되셨나요?

**💡 심화 학습 질문:** Multi-Head Attention에서 여러 개의 "head"를 사용하는 이유는 무엇일까요? 각각 다른 관점에서 정보를 보는 것과 어떤 관련이 있을까요?

=== 연구원 페르소나 ===


Transformer의 Attention 메커니즘은 입력 시퀀스의 모든 위치 간 관계를 병렬적으로 계산하여 문맥 정보를 효율적으로 학습하는 핵심 구조입니다.

## Self-Attention (자기 주의) 메커니즘

**수식:**
```
Attention(Q,K,V) = softmax(QK^T/√d_k)V
```

여기서:
- Q (Query): 질의 행렬
- K (Key): 키 행렬  
- V (Value): 값 행렬
- d_k: 키 벡터의 차원

## 핵심 특징

1. **병렬 처리**: RNN과 달리 모든 위치를 동시에 계산
2. **장거리 의존성**: 시퀀스 길이에 관계없이 직접적 연결
3. **가중 평균**: 각 위치의 중요도를 동적으로 계산

## Multi-Head Attention

```
MultiHead(Q,K,V) = Concat(head_1,...,head_h)W^O
```

- h개의 attention head를 병렬 실행
- 각 head는 서로 다른 표현 부공간에서 attention 계산
- 다양한 유형의 관계 패턴 학습 가능

## 계산 복잡도

- 시간 복잡도: O(n²d) (n: 시퀀스 길이, d: 모델 차원)
- 공간 복잡도: O(n²)

이 메커니즘은 "Attention Is All You Need" (Vaswani et al., 2017)에서 제안되어 현대 NLP의 표준이 되었습니다.


---
## 🔗 실무로 연결하기

프롬프트 엔지니어링은 **'모델 학습 없이' 업무를 자동화**하는 가장 빠른 길입니다.

- **Structured Output(JSON)**: 고객 문의 → `{유형, 긴급도, 부서}` 로 자동 분류해 티켓 시스템에 바로 연결
- **Few-shot**: 회사만의 분류 기준을 예시 몇 개로 학습 없이 주입
- **System Prompt**: 모든 답변을 회사 톤·정책에 맞게 일관 유지


## 💡 심화 도전
1. 동일 질문을 Zero-shot, Few-shot, CoT 세 가지 방식으로 호출해 토큰 수(응답 길이)를 비교해보세요.
2. JSON 응답에 `sentiment_reason` 필드를 추가하도록 System Prompt를 수정하세요.
3. 학생이 '모르겠어요'라고 하면 힌트만 주는 소크라테스식 교육 페르소나를 설계해보세요.


## ✅ 정답 코드

👆 모두 풀고 난 후 확인하세요

```python
# TODO 1
zero_result = claude(zero_shot_template.format(review=review), temperature=0.1)
few_result  = claude(few_shot_template.format(review=review),  temperature=0.1)

# TODO 2
cot_prompt = f'...문제: {problem}...'
claude(cot_prompt, temperature=0.1)

# TODO 3
raw    = claude(user_prompt=text, system_prompt=SYSTEM_JSON, temperature=0.1)
parsed = json.loads(raw)
parsed['sentiment']

# TODO 4
claude(question, system_prompt=SYSTEM_TUTOR)
claude(question, system_prompt=SYSTEM_EXPERT)
```
